In [1]:
# Imports & Config
import os, cv2, h5py, json, torch, warnings, time
import numpy as np
from pathlib import Path
from tqdm import tqdm
from facenet_pytorch import MTCNN
import torch.nn.functional as F
warnings.filterwarnings('ignore')

# ── Paths 
DATASET_ROOT = r'C:\Users\as316\CNN+BiLSTM\CombinedDataset'
OUTPUT_H5    = r'D:\CNN+LSTM Results\faces_9600_380.h5'

# ── Core Config 
IMG_SIZE      = 380   
N_FRAMES      = 15
MAX_REAL      = 4800  
MAX_FAKE      = 4800
MTCNN_BATCH   = 15



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# normalization
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

os.makedirs(os.path.dirname(OUTPUT_H5), exist_ok=True)

print(f'Device     : {DEVICE}')
print(f'IMG_SIZE   : {IMG_SIZE}  (EfficientNetB4)')
print(f'N_FRAMES   : {N_FRAMES}')
print(f'Dataset    : {MAX_REAL} real + {MAX_FAKE} fake = {MAX_REAL+MAX_FAKE} total')
if DEVICE == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM       : {vram:.1f} GB')
    # Estimate space
    raw_gb = (MAX_REAL+MAX_FAKE) * N_FRAMES * IMG_SIZE * IMG_SIZE * 3 * 2 / 1e9
    print(f'Est. H5    : ~{raw_gb*0.55:.1f} GB (LZF compressed float16)')


Device     : cuda
IMG_SIZE   : 380  (EfficientNetB4)
N_FRAMES   : 15
Dataset    : 4800 real + 4800 fake = 9600 total
GPU        : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM       : 6.4 GB
Est. H5    : ~68.6 GB (LZF compressed float16)


In [2]:
# MTCNN Face Detector
detector = MTCNN(
    image_size     = IMG_SIZE,
    margin         = int(IMG_SIZE * 0.15),  
    min_face_size  = 40,
    thresholds     = [0.6, 0.7, 0.8],
    keep_all       = False,
    post_process   = False,              
    select_largest = True,                  
    device         = DEVICE
)
print(f'MTCNN on {DEVICE} | output: {IMG_SIZE}x{IMG_SIZE} | batch={MTCNN_BATCH}')


MTCNN on cuda | output: 380x380 | batch=15


In [3]:
# Processing Functions


def normalize_face(face_tensor):
    face = face_tensor.permute(1, 2, 0).cpu().numpy() / 255.0  
    face = (face - MEAN) / STD
    return face.astype(np.float16)


def fallback_center_crop(frame_rgb):
    h, w  = frame_rgb.shape[:2]
    s     = min(h, w)
    y0    = (h - s) // 2
    x0    = (w - s) // 2
    crop  = frame_rgb[y0:y0+s, x0:x0+s]
    face  = cv2.resize(crop, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    face  = (face / 255.0 - MEAN) / STD
    return face.astype(np.float16)


def sample_frame_indices(total, n=N_FRAMES):
    start = max(0, int(total * 0.05))
    end   = min(total - 1, int(total * 0.95))
    return np.linspace(start, end, n, dtype=int)


def read_frames(video_path, n=N_FRAMES):
    cap   = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    indices = sample_frame_indices(total, n)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            h, w = frame.shape[:2]
            if max(h, w) > 720:
                scale  = 720 / max(h, w)
                frame  = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def process_video(video_path):
    try:
        frames = read_frames(video_path, N_FRAMES)
        if not frames:
            return None

        detected = detector(frames)  
        if not isinstance(detected, list):
            detected = [detected]

        faces = []
        for i, face in enumerate(detected):
            if face is not None:
                faces.append(normalize_face(face))
            else:
                # Fallback: center crop 
                faces.append(fallback_center_crop(frames[i]))

        if not faces:
            return None

        # Pad to exactly N_FRAMES
        while len(faces) < N_FRAMES:
            faces.append(faces[-1])

        return np.stack(faces[:N_FRAMES])  

    except Exception:
        return None


print(f'process_video → ({N_FRAMES}, {IMG_SIZE}, {IMG_SIZE}, 3) float16')

process_video → (15, 380, 380, 3) float16


In [4]:
# Build Balanced Manifest (4800 real + 4800 fake)
import random
random.seed(42) 

VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv'}

def build_balanced_manifest(root, max_real=MAX_REAL, max_fake=MAX_FAKE):
    root     = Path(root)
    real_dir = root / 'Real'
    fake_dir = root / 'Fake'
    manifest = []

    # Collect real
    if real_dir.exists():
        real_files = [p for p in real_dir.rglob('*')
                      if p.suffix.lower() in VIDEO_EXTS]
        random.shuffle(real_files)
        real_files = real_files[:max_real] 
        for p in real_files:
            manifest.append({'path': str(p), 'label': 0, 'method': 'real'})
    else:
        print('WARNING: Real/ not found')

    # Collect fake
    if fake_dir.exists():
        fake_files = [p for p in fake_dir.rglob('*')
                      if p.suffix.lower() in VIDEO_EXTS]
        random.shuffle(fake_files)
        fake_files = fake_files[:max_fake]  # Cap at max_fake
        for p in fake_files:
            manifest.append({'path': str(p), 'label': 1, 'method': 'fake'})
    else:
        print('WARNING: Fake/ not found')

    # Shuffle the combined manifest
    random.shuffle(manifest)

    n_real = sum(1 for m in manifest if m['label'] == 0)
    n_fake = sum(1 for m in manifest if m['label'] == 1)
    print(f'Manifest: {len(manifest)} videos  ({n_real} real | {n_fake} fake)')
    print(f'Balance : {n_real/max(len(manifest),1):.1%} real / {n_fake/max(len(manifest),1):.1%} fake')
    return manifest


manifest = build_balanced_manifest(DATASET_ROOT)
manifest_path = OUTPUT_H5.replace('.h5', '_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest saved: {manifest_path}')


Manifest: 9600 videos  (4800 real | 4800 fake)
Balance : 50.0% real / 50.0% fake
Manifest saved: D:\CNN+LSTM Results\faces_9600_380_manifest.json


In [5]:
# Speed Benchmark (20 videos)
import time

print('Benchmarking 20 videos (sequential GPU pipeline)...')
torch.cuda.empty_cache()

t0 = time.time()
ok = 0
for item in manifest[:20]:
    if process_video(item['path']) is not None:
        ok += 1

elapsed    = time.time() - t0
rate       = 20 / elapsed
total_min  = len(manifest) / rate / 60

print(f'Speed      : {rate:.2f} vid/sec  ({elapsed/20:.2f} sec/vid)')
print(f'Success    : {ok}/20')
print(f'Est. total : {total_min:.0f} min for {len(manifest)} videos')
print(f'GPU memory : {torch.cuda.memory_allocated()/1e6:.0f} MB used')
torch.cuda.empty_cache()


Benchmarking 20 videos (sequential GPU pipeline)...
Speed      : 0.41 vid/sec  (2.44 sec/vid)
Success    : 15/20
Est. total : 390 min for 9600 videos
GPU memory : 11 MB used


In [6]:
# preprocessing using Sequential

failed  = []
n       = len(manifest)
t_start = time.time()

torch.cuda.empty_cache()

with h5py.File(OUTPUT_H5, 'w') as hf:

    # Pre-allocate full dataset
    faces_ds = hf.create_dataset(
        'faces',
        shape       = (n, N_FRAMES, IMG_SIZE, IMG_SIZE, 3),
        dtype       = 'float16',
        chunks      = (1, N_FRAMES, IMG_SIZE, IMG_SIZE, 3),
        compression = 'lzf',
        shuffle     = True
    )
    labels_ds  = hf.create_dataset('labels',  shape=(n,), dtype='uint8')
    methods_ds = hf.create_dataset('methods', shape=(n,), dtype=h5py.string_dtype())
    paths_ds   = hf.create_dataset('paths',   shape=(n,), dtype=h5py.string_dtype())

    valid = 0

    for item in tqdm(manifest, desc='Preprocessing', unit='vid',
                     dynamic_ncols=True):
        result = process_video(item['path'])

        if result is None:
            failed.append(item['path'])
            continue

        faces_ds[valid]   = result
        labels_ds[valid]  = item['label']
        methods_ds[valid] = item['method']
        paths_ds[valid]   = item['path']
        valid += 1

        # Periodic GPU memory cleanup
        if valid % 200 == 0:
            torch.cuda.empty_cache()

    # Store metadata
    hf.attrs['n_valid']    = valid
    hf.attrs['n_failed']   = len(failed)
    hf.attrs['img_size']   = IMG_SIZE
    hf.attrs['n_frames']   = N_FRAMES
    hf.attrs['model']      = 'EfficientNetB4+BiLSTM'
    hf.attrs['dtype']      = 'float16'
    hf.attrs['n_real']     = sum(1 for m in manifest[:valid] if m['label']==0)
    hf.attrs['n_fake']     = sum(1 for m in manifest[:valid] if m['label']==1)

total_min = (time.time() - t_start) / 60
h5_gb     = os.path.getsize(OUTPUT_H5) / 1e9
print(f'\n{"="*50}')
print(f'DONE')
print(f'{"="*50}')
print(f'Saved     : {valid}/{n}')
print(f'Failed    : {len(failed)}')
print(f'Time      : {total_min:.1f} min')
print(f'File size : {h5_gb:.2f} GB')
print(f'Output    : {OUTPUT_H5}')

failed_path = OUTPUT_H5.replace('.h5', '_failed.json')
with open(failed_path, 'w') as f:
    json.dump(failed, f, indent=2)
print(f'Failed list: {failed_path}')


Preprocessing: 100%|████████████████████████████████████████████████████████████| 9600/9600 [5:36:39<00:00,  2.10s/vid]


DONE
Saved     : 8891/9600
Failed    : 709
Time      : 336.7 min
File size : 47.78 GB
Output    : D:\CNN+LSTM Results\faces_9600_380.h5
Failed list: D:\CNN+LSTM Results\faces_9600_380_failed.json


In [7]:
# Verify Output

def verify_h5(path):
    with h5py.File(path, 'r') as hf:
        n      = int(hf.attrs.get('n_valid', 0))
        labels = hf['labels'][:n]
        n_real = int((labels == 0).sum())
        n_fake = int((labels == 1).sum())
        print('='*50)
        print('HDF5 Verification')
        print('='*50)
        print(f"Model      : {hf.attrs.get('model','?')}")
        print(f"IMG_SIZE   : {hf.attrs.get('img_size','?')}")
        print(f"N_FRAMES   : {hf.attrs.get('n_frames','?')}")
        print(f"Shape      : {hf['faces'].shape}")
        print(f"Dtype      : {hf['faces'].dtype}")
        print(f"Valid      : {n}  (real={n_real} | fake={n_fake})")
        print(f"Balance    : {n_real/max(n,1):.1%} / {n_fake/max(n,1):.1%}")
        if n > 0:
            s  = hf['faces'][0].astype(np.float32)
            ok = s.min() < -0.5 and s.max() > 0.5
            print(f"Sample[0]  : min={s.min():.3f} max={s.max():.3f}")
            print(f"Norm check : {'PASSED' if ok else 'WARNING - check normalization'}")
        print(f"File size  : {os.path.getsize(path)/1e9:.2f} GB")
        print('='*50)


if os.path.exists(OUTPUT_H5):
    verify_h5(OUTPUT_H5)
else:
    print('Run Cell 6 first.')


HDF5 Verification
Model      : EfficientNetB4+BiLSTM
IMG_SIZE   : 380
N_FRAMES   : 15
Shape      : (9600, 15, 380, 380, 3)
Dtype      : float16
Valid      : 8891  (real=4374 | fake=4517)
Balance    : 49.2% / 50.8%
Sample[0]  : min=-2.102 max=1.735
Norm check : PASSED
File size  : 47.78 GB
